# Voice Interview Agent

**30 min project**

Build a progressive voice-based mock interview agent for an AI Agents Engineer role.

**Concepts:** VoicePipeline, audio input, memory, tools, handoffs, custom voice workflows, TTS tuning


In [ ]:
!pip install -q openai-agents[voice] python-dotenv numpy sounddevice

In [6]:
from dotenv import load_dotenv

load_dotenv("../../.env")


True

## 1. Setup and Audio Helpers


In [7]:
import asyncio
import numpy as np
import sounddevice as sd
from typing import AsyncIterator

from agents import Agent, CodeInterpreterTool, Runner, WebSearchTool
from agents.extensions.handoff_prompt import prompt_with_handoff_instructions
from agents.voice import (
    AudioInput,
    SingleAgentVoiceWorkflow,
    SingleAgentWorkflowCallbacks,
    StreamedAudioInput,
    TTSModelSettings,
    VoicePipeline,
    VoicePipelineConfig,
    VoiceWorkflowBase,
    VoiceWorkflowHelper,
)

SAMPLE_RATE = 24000
STOP_PHRASES = {"stop", "exit", "quit", "goodbye", "end interview", "that's all"}


def record_audio(duration: int = 5, sample_rate: int = SAMPLE_RATE) -> np.ndarray:
    print(f"Recording for {duration} seconds...")
    audio = sd.rec(int(duration * sample_rate), samplerate=sample_rate, channels=1, dtype="int16")
    sd.wait()
    return audio.flatten()


def play_audio(chunks: list[np.ndarray], sample_rate: int = SAMPLE_RATE) -> None:
    audio = np.concatenate(chunks)
    sd.play(audio, samplerate=sample_rate)
    sd.wait()


async def play_pipeline_result(result) -> None:
    chunks = []
    async for event in result.stream():
        if event.type == "voice_stream_event_audio":
            chunks.append(event.data)
    if chunks:
        play_audio(chunks)


## 2. Single-Turn Voice Agent

A `VoicePipeline` turns recorded audio into text, runs an agent workflow, then turns the response back into audio.


In [9]:
single_turn_agent = Agent(
    name="SingleTurnVoiceAssistant",
    model="gpt-4o-mini",
    instructions="You are a concise voice assistant. Answer in one short sentence.",
)

async def single_turn_voice_demo(duration: int = 5):
    workflow = SingleAgentVoiceWorkflow(single_turn_agent)
    pipeline = VoicePipeline(workflow=workflow)

    audio = record_audio(duration)
    result = await pipeline.run(AudioInput(buffer=audio))
    await play_pipeline_result(result)

# Run this when you want to test one voice turn.
await single_turn_voice_demo(duration=5)


Recording for 5 seconds...


## 3. Multi-Turn Memory

`SingleAgentVoiceWorkflow` stores the conversation history internally when the same workflow instance is reused.


In [10]:
class TranscriptLogger(SingleAgentWorkflowCallbacks):
    def __init__(self):
        self.last_transcription = ""

    def on_run(self, workflow: SingleAgentVoiceWorkflow, transcription: str) -> None:
        self.last_transcription = transcription
        print(f"You said: {transcription}")

memory_agent = Agent(
    name="MemoryVoiceAssistant",
    model="gpt-4o-mini",
    instructions="""You are a concise voice assistant.
Remember earlier turns and answer in no more than two short sentences.""",
)

async def multi_turn_memory_demo(duration: int = 5, max_turns: int = 3):
    callbacks = TranscriptLogger()
    workflow = SingleAgentVoiceWorkflow(memory_agent, callbacks=callbacks)
    pipeline = VoicePipeline(workflow=workflow)

    for turn in range(1, max_turns + 1):
        command = input(f"Turn {turn}: press Enter to speak, or type 'q' to stop: ")
        if command.lower() in {"q", "quit", "exit"}:
            break

        audio = record_audio(duration)
        result = await pipeline.run(AudioInput(buffer=audio))
        await play_pipeline_result(result)

        if any(phrase in callbacks.last_transcription.lower() for phrase in STOP_PHRASES):
            print("Conversation ended by voice command.")
            break

# Ask two related questions to test memory.
await multi_turn_memory_demo(duration=5, max_turns=3)


Recording for 5 seconds...
You said: Hello, my name is Ishan.
Recording for 5 seconds...
You said: What is my name?


## 4. Official SDK Tools

Use hosted OpenAI tools instead of toy local tools:

- `WebSearchTool` for current market/interview context
- `CodeInterpreterTool` for rubric calculations and score aggregation


In [11]:
resume_text = """
Ishan Dutta
Software engineer building AI products, agent workflows, and developer tools.
Experience with Python, TypeScript, OpenAI APIs, multi-agent systems, RAG, MCP, evaluation, and workshop teaching.
Built resume analyzers, research agents, GitHub Copilot-style assistants, and voice prototypes.
"""

job_description = """
AI Agents Engineer
We need an engineer who can design reliable AI agent workflows using LLMs, tools, memory, orchestration, evaluation, and production observability.
The role requires Python, agent frameworks, prompt design, tool calling, retrieval, guardrails, and clear communication with product teams.
"""

candidate_context = f"""
Candidate resume:
{resume_text}

Target job description:
{job_description}
"""

official_interview_tools = [
    WebSearchTool(),
    CodeInterpreterTool(
        tool_config={"type": "code_interpreter", "container": {"type": "auto"}},
    ),
]


In [12]:
tool_interviewer_agent = Agent(
    name="ToolInterviewer",
    model="gpt-5.5",
    instructions=f"""You are a mock interviewer for an AI Agents Engineer role.

Use the official hosted tools when useful:
- Use WebSearchTool for current AI agents interview expectations and market context.
- Use CodeInterpreterTool for scoring rubrics or aggregating evaluation signals.

Candidate context:
{candidate_context}

Ask one very easy question at a time. Keep every spoken response under 35 words.""",
    tools=official_interview_tools,
)

async def personalized_voice_question(duration: int = 5):
    workflow = SingleAgentVoiceWorkflow(tool_interviewer_agent)
    pipeline = VoicePipeline(workflow=workflow)

    audio = record_audio(duration)
    result = await pipeline.run(AudioInput(buffer=audio))
    await play_pipeline_result(result)

# Say: "Start the AI agents interview."
await personalized_voice_question(duration=5)


Recording for 5 seconds...


## 5. Multi-Agent Interview System

Handoffs let a voice workflow move between specialists while `SingleAgentVoiceWorkflow` tracks the current agent.


In [ ]:
voice_response_rules = f"""
You are speaking aloud. Keep every response under 35 words.
Ask one easy question at a time. Do not use bullet points.

Candidate context:
{candidate_context}
"""

feedback_agent = Agent(
    name="FeedbackAgent",
    handoff_description="Gives the final interview scorecard.",
    model="gpt-5.5",
    instructions=voice_response_rules + """
Use CodeInterpreterTool if you need to calculate or aggregate a score.
Create a short final scorecard for the AI Agents Engineer interview.
Mention one strength, one improvement area, and an overall score out of 5.
""",
    tools=official_interview_tools,
)

questioner_agent = Agent(
    name="QuestionerAgent",
    handoff_description="Asks easy AI agents interview questions and evaluates answers.",
    model="gpt-5.5",
    instructions=prompt_with_handoff_instructions(voice_response_rules + """
You ask beginner-friendly AI Agents Engineer questions.
Use WebSearchTool if you need current role expectations.
Use CodeInterpreterTool if you need to compute a score.
For each candidate answer, give one short evaluation sentence, then ask the next easy question.
If the candidate wants to stop, hand off to FeedbackAgent.
"""),
    tools=official_interview_tools,
    handoffs=[feedback_agent],
)

greeter_agent = Agent(
    name="GreeterAgent",
    handoff_description="Starts the interview and routes to the questioner.",
    model="gpt-5.5",
    instructions=prompt_with_handoff_instructions(voice_response_rules + """
Greet the candidate, explain this will be an easy AI Agents Engineer interview, then hand off to QuestionerAgent.
"""),
    tools=[WebSearchTool()],
    handoffs=[questioner_agent],
)

async def handoff_voice_demo(duration: int = 5, turns: int = 3):
    workflow = SingleAgentVoiceWorkflow(greeter_agent)
    pipeline = VoicePipeline(workflow=workflow)

    for turn in range(1, turns + 1):
        command = input(f"Turn {turn}: press Enter to speak, or type 'q' to stop: ")
        if command.lower() in {"q", "quit", "exit"}:
            break
        audio = record_audio(duration)
        result = await pipeline.run(AudioInput(buffer=audio))
        await play_pipeline_result(result)

# await handoff_voice_demo(duration=5, turns=3)


## 6. Custom Voice Workflow

Subclass `VoiceWorkflowBase` when you want explicit control over history, greetings, stop words, and final feedback.


In [ ]:
class InterviewVoiceWorkflow(VoiceWorkflowBase):
    def __init__(self, starting_agent: Agent, final_agent: Agent, max_questions: int = 4):
        self.starting_agent = starting_agent
        self.final_agent = final_agent
        self.current_agent = starting_agent
        self.max_questions = max_questions
        self.question_count = 0
        self.history = []
        self.ended = False

    async def on_start(self) -> AsyncIterator[str]:
        yield "Welcome to your AI Agents Engineer mock interview. I will ask easy questions. Say stop when you want to finish."

    async def run(self, transcription: str) -> AsyncIterator[str]:
        print(f"You said: {transcription}")
        self.history.append({"role": "user", "content": transcription})

        should_finish = any(phrase in transcription.lower() for phrase in STOP_PHRASES)
        if should_finish or self.question_count >= self.max_questions:
            self.current_agent = self.final_agent
            self.history.append({"role": "user", "content": "Please end the interview with a short spoken scorecard."})
            self.ended = True
        else:
            self.question_count += 1

        result = Runner.run_streamed(self.current_agent, self.history)
        async for text in VoiceWorkflowHelper.stream_text_from(result):
            yield text

        self.history = result.to_input_list()
        self.current_agent = result.last_agent

workflow_preview = InterviewVoiceWorkflow(greeter_agent, feedback_agent, max_questions=3)


## 7. Voice Tuning

`TTSModelSettings` controls voice, pace, and delivery style for the spoken output.


In [ ]:
interviewer_tts_settings = TTSModelSettings(
    voice="nova",
    speed=1.1,
    instructions="""
Voice: friendly, clear, and professional.
Pacing: concise and slightly brisk.
Delivery: keep each answer under about ten seconds. Pause briefly before each question.
Emotion: supportive interviewer, not dramatic.
""",
)

interview_voice_config = VoicePipelineConfig(
    workflow_name="AI Agents Engineer Voice Interview",
    tts_settings=interviewer_tts_settings,
)


## 8. Full Live Interview

This combines tools, handoffs, memory, custom workflow control, and voice tuning.


In [ ]:
async def run_full_interview(duration: int = 8, max_questions: int = 4):
    workflow = InterviewVoiceWorkflow(greeter_agent, feedback_agent, max_questions=max_questions)
    pipeline = VoicePipeline(workflow=workflow, config=interview_voice_config)

    print("AI Agents Engineer voice interview")
    print("First say: start interview")
    print("Then answer each spoken question.")
    print("Say 'stop' in your answer for a spoken final scorecard.")
    print("Type 'q' before a turn to end without final feedback.\n")

    turn = 1
    while not workflow.ended:
        command = input(f"Turn {turn}: press Enter to speak, or type 'q' to stop: ")
        if command.lower() in {"q", "quit", "exit"}:
            break

        audio = record_audio(duration)
        result = await pipeline.run(AudioInput(buffer=audio))
        await play_pipeline_result(result)
        turn += 1

# Main workshop demo.
# await run_full_interview(duration=8, max_questions=4)


In [ ]:
async def stream_microphone_until_enter(streamed_input: StreamedAudioInput, sample_rate: int = SAMPLE_RATE):
    loop = asyncio.get_running_loop()

    def callback(indata, frames, time, status):
        chunk = indata.copy().reshape(-1)
        asyncio.run_coroutine_threadsafe(streamed_input.add_audio(chunk), loop)

    print("Streaming microphone. Speak naturally; press Enter to end the session.")
    with sd.InputStream(samplerate=sample_rate, channels=1, dtype="int16", callback=callback):
        await asyncio.to_thread(input)

    await streamed_input.add_audio(None)

async def run_streamed_interview(max_questions: int = 4):
    workflow = InterviewVoiceWorkflow(greeter_agent, feedback_agent, max_questions=max_questions)
    streamed_input = StreamedAudioInput()
    pipeline = VoicePipeline(workflow=workflow, config=interview_voice_config)

    result = await pipeline.run(streamed_input)
    recorder_task = asyncio.create_task(stream_microphone_until_enter(streamed_input))

    player = sd.OutputStream(samplerate=SAMPLE_RATE, channels=1, dtype=np.int16)
    player.start()

    async for event in result.stream():
        if event.type == "voice_stream_event_audio":
            player.write(event.data)
        elif event.type == "voice_stream_event_lifecycle":
            print(event.event)

    await recorder_task
    player.stop()
    player.close()

# Advanced streaming demo.
# await run_streamed_interview(max_questions=4)
